## 1. Import Libraries and Load Data

In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
sns.set_theme(style='whitegrid')


In [49]:
df = pd.read_csv('dataset/fifa_player_performance_market_value.csv')
df.head()

,player_id,player_name,age,nationality,club,position,overall_rating,potential_rating,matches_played,goals,assists,minutes_played,market_value_million_eur,contract_years_left,injury_prone,transfer_risk_level
0,1,Player_1,23,Germany,Liverpool,ST,65,87,8,6,14,2976,122.51,3,No,Low
1,2,Player_2,36,England,FC Barcelona,ST,90,76,19,3,18,2609,88.47,5,No,High
2,3,Player_3,31,France,Juventus,RB,75,91,34,12,15,1158,20.24,3,No,Medium
3,4,Player_4,27,Portugal,Manchester City,LW,90,86,35,18,13,145,164.29,0,Yes,Medium
4,5,Player_5,24,Brazil,Liverpool,CDM,84,96,41,6,6,2226,121.34,4,No,Low


## 2. Drop Feature

In [50]:
df = df.drop(columns=['player_id','player_name','club'],errors='ignore')
df.head()

,age,nationality,position,overall_rating,potential_rating,matches_played,goals,assists,minutes_played,market_value_million_eur,contract_years_left,injury_prone,transfer_risk_level
0,23,Germany,ST,65,87,8,6,14,2976,122.51,3,No,Low
1,36,England,ST,90,76,19,3,18,2609,88.47,5,No,High
2,31,France,RB,75,91,34,12,15,1158,20.24,3,No,Medium
3,27,Portugal,LW,90,86,35,18,13,145,164.29,0,Yes,Medium
4,24,Brazil,CDM,84,96,41,6,6,2226,121.34,4,No,Low


## 3.Feature Engineering

In [51]:
df['growth_dynamic'] = (df['potential_rating'] - df['overall_rating']).abs()
df['growth_percentage'] = (df['potential_rating'] - df['overall_rating'] / df['overall_rating'])
df['avg_minutes_per_match'] = df['minutes_played'] / df['matches_played']

df['goals_per_90'] = ((df['goals']/df['minutes_played']) * 90)
df['assists_per_90'] = ((df['goals']/df['assists']) * 90)
df['points_per_90'] = ((df['goals']+df['assists']/df['minutes_played']) * 90)
df['total_contributions'] = df['goals']+df['assists']

df['contract_expiry'] = df['contract_years_left'] + df['age']
df['contract_ration'] = df['contract_years_left']/df['age']

df.head(5)

,age,nationality,position,overall_rating,potential_rating,matches_played,goals,assists,minutes_played,market_value_million_eur,...,transfer_risk_level,growth_dynamic,growth_percentage,avg_minutes_per_match,goals_per_90,assists_per_90,points_per_90,total_contributions,contract_expiry,contract_ration
0,23,Germany,ST,65,87,8,6,14,2976,122.51,...,Low,22,86.0,372.000000,0.181452,38.571429,540.423387,20,26,0.130435
1,36,England,ST,90,76,19,3,18,2609,88.47,...,High,14,75.0,137.315789,0.103488,15.000000,270.620928,21,41,0.138889
2,31,France,RB,75,91,34,12,15,1158,20.24,...,Medium,16,90.0,34.058824,0.932642,72.000000,1081.165803,27,34,0.096774
3,27,Portugal,LW,90,86,35,18,13,145,164.29,...,Medium,4,85.0,4.142857,11.172414,124.615385,1628.068966,31,27,0.000000
4,24,Brazil,CDM,84,96,41,6,6,2226,121.34,...,Low,12,95.0,54.292683,0.242588,90.000000,540.242588,12,28,0.166667


In [52]:
conditions = [
    (df['age'] < df['age'].mean()) & (df['contract_years_left'] > df['contract_years_left'].mean()),
    (df['age'] > df['age'].mean()) & (df['contract_years_left'] <= df['contract_years_left'].mean()),
    (df['age'].between(df['age'].mean(),df['age'].max())) & (df['contract_years_left'] > df['contract_years_left'].mean()),
]
choices = ['Young Prospect', 'Veteran/Short-term', 'Stable Peak']
df['career_phase'] = np.select(conditions,choices,default='Other / Transitional')

conditions = [
    (df['avg_minutes_per_match'] >= 70),
    (df['avg_minutes_per_match'] >= 30) & (df['avg_minutes_per_match'] <= 69),
    (df['avg_minutes_per_match'] < 30)
]
choices = ['Key Player / Regular Starter', 'Squad Player / Rotation', 'Impact Sub / Benchwarmer']
df['player_role_cluster'] = np.select(conditions, choices, default='Unknown')


rating_diff = df['potential_rating'] - df['overall_rating']
conditions = [
    (df['overall_rating'] < 75) & (df['potential_rating'] >= 85),
    (rating_diff >= 15),
    (df['overall_rating'] >= 85) & (df['potential_rating'] >= 85)
]
choices = ['Superstar Candidate', 'Hidden Gem', 'Elite Status']
df['investment_class'] = np.select(conditions, choices, default='Regular Player')


## 4. Feature Categorical Encoding

In [53]:
label = LabelEncoder()
feature_categori = df.select_dtypes(include='object').columns

for col in feature_categori:
    df[col] = label.fit_transform(df[col])

df[feature_categori].head()

,nationality,position,injury_prone,transfer_risk_level,career_phase,player_role_cluster,investment_class
0,4,8,0,1,3,1,3
1,2,8,0,0,1,1,2
2,3,6,0,2,1,2,1
3,6,5,1,2,0,0,0
4,1,1,0,1,3,2,2


## 5. Save Cleaned Data

In [54]:
df.to_csv("dataset/fifa_player_performance_market_value_cleaned.csv", index=False)
print("Cleaned dataset saved to data/titanic_cleaned.csv")
print(f"Final shape: {df.shape}")
df.head()

Cleaned dataset saved to data/titanic_cleaned.csv
Final shape: (2800, 25)


,age,nationality,position,overall_rating,potential_rating,matches_played,goals,assists,minutes_played,market_value_million_eur,...,avg_minutes_per_match,goals_per_90,assists_per_90,points_per_90,total_contributions,contract_expiry,contract_ration,career_phase,player_role_cluster,investment_class
0,23,4,8,65,87,8,6,14,2976,122.51,...,372.000000,0.181452,38.571429,540.423387,20,26,0.130435,3,1,3
1,36,2,8,90,76,19,3,18,2609,88.47,...,137.315789,0.103488,15.000000,270.620928,21,41,0.138889,1,1,2
2,31,3,6,75,91,34,12,15,1158,20.24,...,34.058824,0.932642,72.000000,1081.165803,27,34,0.096774,1,2,1
3,27,6,5,90,86,35,18,13,145,164.29,...,4.142857,11.172414,124.615385,1628.068966,31,27,0.000000,0,0,0
4,24,1,1,84,96,41,6,6,2226,121.34,...,54.292683,0.242588,90.000000,540.242588,12,28,0.166667,3,2,2
